# ЛР-1. Бенчмарк для отчёта — вариант 3 (поэлементное умножение векторов)

Сбор данных для таблицы «Результаты» в отчёте.

**Runtime → Change runtime type → GPU (T4)**, затем **Runtime → Run all**.

Время меряется только ядро (CUDA-события), среднее по 10 запускам (3 прогревочных),
плюс однопоточная CPU-версия (g++ -O2, время самого цикла).

In [ ]:
!nvidia-smi
import os
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
!nvcc --version
!g++ --version | head -1
!head -2 /etc/os-release


## Свойства GPU (для раздела «Программное и аппаратное обеспечение»)

In [ ]:
%%writefile query.cu
#include <cstdio>
#include <cuda_runtime.h>

int main(void)
{
    int count = 0;
    if (cudaGetDeviceCount(&count) != cudaSuccess || count < 1) {
        fprintf(stderr, "ERROR: no CUDA device\n");
        return 1;
    }
    cudaDeviceProp p;
    if (cudaGetDeviceProperties(&p, 0) != cudaSuccess) {
        fprintf(stderr, "ERROR: cudaGetDeviceProperties failed\n");
        return 1;
    }
    printf("Name                    : %s\n", p.name);
    printf("Compute capability      : %d.%d\n", p.major, p.minor);
    printf("Multiprocessors count   : %d\n", p.multiProcessorCount);
    printf("Total global memory     : %zu\n", p.totalGlobalMem);
    printf("Shared memory per block : %zu\n", p.sharedMemPerBlock);
    printf("Total constant memory   : %zu\n", p.totalConstMem);
    printf("Registers per block     : %d\n", p.regsPerBlock);
    printf("Warp size               : %d\n", p.warpSize);
    printf("Max threads per block   : (%d, %d, %d)\n", p.maxThreadsDim[0], p.maxThreadsDim[1], p.maxThreadsDim[2]);
    printf("Max block               : (%d, %d, %d)\n", p.maxGridSize[0], p.maxGridSize[1], p.maxGridSize[2]);
    return 0;
}


In [ ]:
import subprocess
r = subprocess.run('nvcc -o query query.cu', shell=True, capture_output=True, text=True)
assert r.returncode == 0, 'query compile failed:\n' + r.stderr
r = subprocess.run(['./query'], capture_output=True, text=True)
print(r.stdout, end='')
if r.stderr:
    print('STDERR:', r.stderr)


## Бенчмарк ядра (grid-stride умножение, как в lab1.cu)

In [ ]:
%%writefile bench.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define RUNS 10
#define WARMUP 3

__global__ void mulKernel(const double *a, const double *b, double *c, long long n)
{
    long long stride = (long long)gridDim.x * blockDim.x;
    for (long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        c[i] = a[i] * b[i];
}

static void fail(const char *msg)
{
    fprintf(stderr, "ERROR: %s\n", msg);
    exit(1);
}

struct Config {
    int blocks;
    int threads;
};

int main(void)
{
    const long long sizes[3] = {1000LL, 1000000LL, (1LL << 25) - 1};
    Config configs[] = {
        {1, 32}, {1, 64}, {1, 128}, {1, 256}, {1, 512}, {1, 1024},
        {32, 32}, {1024, 1}, {32, 256}, {128, 64}, {256, 256},
        {1024, 256}, {1024, 512}, {1024, 1024},
    };
    const int numConfigs = (int)(sizeof(configs) / sizeof(configs[0]));

    cudaError_t err;
    int devCount = 0;
    if ((err = cudaGetDeviceCount(&devCount)) != cudaSuccess || devCount < 1)
        fail("no CUDA device");

    cudaEvent_t start, stop;
    if ((err = cudaEventCreate(&start)) != cudaSuccess)
        fail("cudaEventCreate failed");
    if ((err = cudaEventCreate(&stop)) != cudaSuccess)
        fail("cudaEventCreate failed");

    for (int s = 0; s < 3; s++) {
        long long n = sizes[s];
        size_t bytes = (size_t)n * sizeof(double);

        double *a = (double *)malloc(bytes);
        double *b = (double *)malloc(bytes);
        double *c = (double *)malloc(bytes);
        if (!a || !b || !c)
            fail("host malloc failed");
        for (long long i = 0; i < n; i++) {
            a[i] = 1.0 + (double)(i % 1000) / 1000.0;
            b[i] = 1.0 + (double)((i * 7) % 997) / 997.0;
        }

        double *da = NULL, *db = NULL, *dc = NULL;
        if ((err = cudaMalloc((void **)&da, bytes)) != cudaSuccess)
            fail("cudaMalloc failed");
        if ((err = cudaMalloc((void **)&db, bytes)) != cudaSuccess)
            fail("cudaMalloc failed");
        if ((err = cudaMalloc((void **)&dc, bytes)) != cudaSuccess)
            fail("cudaMalloc failed");
        if ((err = cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice)) != cudaSuccess)
            fail("cudaMemcpy failed");
        if ((err = cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice)) != cudaSuccess)
            fail("cudaMemcpy failed");

        for (int k = 0; k < numConfigs; k++) {
            dim3 grid(configs[k].blocks);
            dim3 block(configs[k].threads);
            for (int w = 0; w < WARMUP; w++)
                mulKernel<<<grid, block>>>(da, db, dc, n);
            if ((err = cudaGetLastError()) != cudaSuccess)
                fail("kernel launch failed");
            double total = 0.0;
            for (int r = 0; r < RUNS; r++) {
                if ((err = cudaEventRecord(start)) != cudaSuccess)
                    fail("cudaEventRecord failed");
                mulKernel<<<grid, block>>>(da, db, dc, n);
                if ((err = cudaEventRecord(stop)) != cudaSuccess)
                    fail("cudaEventRecord failed");
                if ((err = cudaEventSynchronize(stop)) != cudaSuccess)
                    fail("cudaEventSynchronize failed");
                float ms = 0.0f;
                if ((err = cudaEventElapsedTime(&ms, start, stop)) != cudaSuccess)
                    fail("cudaEventElapsedTime failed");
                total += (double)ms;
            }
            printf("BENCH %d %d %lld %.6f\n", configs[k].blocks, configs[k].threads, n, total / RUNS);
        }

        if ((err = cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost)) != cudaSuccess)
            fail("cudaMemcpy failed");
        for (long long i = 0; i < n; i++) {
            if (c[i] != a[i] * b[i]) {
                fprintf(stderr, "ERROR: verify failed at %lld\n", i);
                return 1;
            }
        }
        printf("VERIFY %lld OK\n", n);

        cudaFree(da);
        cudaFree(db);
        cudaFree(dc);
        free(a);
        free(b);
        free(c);
    }

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    return 0;
}


In [ ]:
import subprocess
r = subprocess.run('nvcc -O3 -o bench bench.cu', shell=True, capture_output=True, text=True)
assert r.returncode == 0, 'bench compile failed:\n' + r.stderr
r = subprocess.run(['./bench'], capture_output=True, text=True)
print('rc:', r.returncode)
if r.stderr:
    print('STDERR:', r.stderr[:1000])
open('bench_out.txt', 'w').write(r.stdout)
print(r.stdout[-400:], end='')
assert r.returncode == 0 and r.stdout.count('VERIFY') == 3, 'bench failed'
print('\nBENCH DONE')


## CPU-версия (g++ -O2, один поток, время цикла)

In [ ]:
%%writefile cpu.cpp
#include <cstdio>
#include <cstdlib>
#include <chrono>

#define RUNS 10

int main(void)
{
    const long long sizes[3] = {1000LL, 1000000LL, (1LL << 25) - 1};
    for (int s = 0; s < 3; s++) {
        long long n = sizes[s];
        double *a = (double *)malloc((size_t)n * sizeof(double));
        double *b = (double *)malloc((size_t)n * sizeof(double));
        double *c = (double *)malloc((size_t)n * sizeof(double));
        if (!a || !b || !c) {
            fprintf(stderr, "ERROR: malloc failed\n");
            return 1;
        }
        for (long long i = 0; i < n; i++) {
            a[i] = 1.0 + (double)(i % 1000) / 1000.0;
            b[i] = 1.0 + (double)((i * 7) % 997) / 997.0;
        }
        double total = 0.0;
        for (int r = 0; r < RUNS; r++) {
            auto t0 = std::chrono::high_resolution_clock::now();
            for (long long i = 0; i < n; i++)
                c[i] = a[i] * b[i];
            auto t1 = std::chrono::high_resolution_clock::now();
            total += std::chrono::duration<double, std::milli>(t1 - t0).count();
        }
        printf("CPU %lld %.6f %.15f\n", n, total / RUNS, c[0] + c[n - 1]);
        free(a);
        free(b);
        free(c);
    }
    return 0;
}


In [ ]:
import subprocess
r = subprocess.run('g++ -O2 -o cpu cpu.cpp', shell=True, capture_output=True, text=True)
assert r.returncode == 0, 'cpu compile failed:\n' + r.stderr
r = subprocess.run(['./cpu'], capture_output=True, text=True)
print('rc:', r.returncode)
if r.stderr:
    print('STDERR:', r.stderr[:1000])
open('cpu_out.txt', 'w').write(r.stdout)
print(r.stdout, end='')
assert r.returncode == 0 and r.stdout.count('CPU') == 3, 'cpu failed'
print('CPU DONE')


## Итоговая таблица (формат отчёта) + пропускная способность

Если файлы результатов неполны — ячейка сама перезапустит бинарники и покажет их вывод.

In [ ]:
import os, subprocess

N1, N2, N3 = 1000, 1000000, 33554431
order = [(1, 32), (1, 64), (1, 128), (1, 256), (1, 512), (1, 1024),
         (32, 32), (1024, 1), (32, 256), (128, 64), (256, 256),
         (1024, 256), (1024, 512), (1024, 1024)]

def ensure_output(name, binary, marker):
    need_rerun = True
    if os.path.exists(name):
        content = open(name).read()
        if marker in content:
            need_rerun = False
    if need_rerun:
        print(f'>>> {name} отсутствует или неполон, перезапускаю ./{binary}')
        r = subprocess.run(['./' + binary], capture_output=True, text=True)
        print(f'>>> rc={r.returncode}, stderr={r.stderr[:500]!r}')
        open(name, 'w').write(r.stdout)
    print(f'--- {name} ---')
    print(open(name).read(), end='')
    print('--- конец ---')

ensure_output('bench_out.txt', 'bench', 'VERIFY %d OK' % N3)
ensure_output('cpu_out.txt', 'cpu', 'CPU %d ' % N3)

gpu = {}
for line in open('bench_out.txt'):
    p = line.split()
    if not p or p[0] != 'BENCH':
        continue
    b, t, n, ms = int(p[1]), int(p[2]), int(p[3]), float(p[4])
    gpu.setdefault((b, t), {})[n] = ms

cpu = {}
for line in open('cpu_out.txt'):
    p = line.split()
    if not p or p[0] != 'CPU':
        continue
    cpu[int(p[1])] = float(p[2])

print('распознано конфигураций GPU:', len(gpu), '| размеров CPU:', sorted(cpu.keys()))
assert set(gpu.keys()) == set(order), 'bench output incomplete'
assert set(cpu.keys()) == {N1, N2, N3}, 'cpu output incomplete'

def fmt_us(ms):
    return ('%.1f' % (ms * 1000.0)).replace('.', ',')

def fmt_ms(ms):
    return ('%.2f' % ms).replace('.', ',')

def fmt_threads(v):
    return f"{v:,}".replace(",", " ")

rows = []
for (b, t) in order:
    d = gpu[(b, t)]
    rows.append((f'<<<{b}, {t}>>>', fmt_threads(b * t), fmt_us(d[N1]), fmt_ms(d[N2]), fmt_ms(d[N3])))
rows.append(('CPU (g++ -O2)', '1 поток', fmt_us(cpu[N1]), fmt_ms(cpu[N2]), fmt_ms(cpu[N3])))

print()
print('| Конфигурация (блоки × нити) | Всего нитей | n = 10³, мкс | n = 10⁶, мс | n = 2²⁵ − 1, мс |')
print('|---|---|---|---|---|')
for r in rows:
    print('| ' + ' | '.join(r) + ' |')

print()
bytes_moved = 3 * 8 * N3
best_cfg = min(order, key=lambda k: gpu[k][N3])
best_ms = gpu[best_cfg][N3]
work_ms = gpu[(256, 256)][N3]
print(f'Пропускная способность (n = 2^25-1, 3*8*n = {bytes_moved/1e6:.0f} МБ):')
print(f'  лучшая конфигурация <<<{best_cfg[0]}, {best_cfg[1]}>>>: {best_ms:.2f} мс -> {bytes_moved/(best_ms*1e-3)/1e9:.1f} ГБ/с')
print(f'  рабочая конфигурация  <<<256, 256>>>: {work_ms:.2f} мс -> {bytes_moved/(work_ms*1e-3)/1e9:.1f} ГБ/с')
print(f'CPU на n = 2^25-1: {cpu[N3]:.2f} мс')
print(f'Ускорение рабочей конфигурации против CPU: n=10^6: {cpu[N2]/gpu[(256,256)][N2]:.1f}x, n=2^25-1: {cpu[N3]/work_ms:.1f}x')
print(f'Ускорение лучшей конфигурации против CPU: n=10^6: {cpu[N2]/gpu[best_cfg][N2]:.1f}x, n=2^25-1: {cpu[N3]/best_ms:.1f}x')


## Что прислать для отчёта

1. Вывод ячейки с `nvidia-smi` / `nvcc --version` / ОС (окружение).
2. Вывод `./query` (свойства GPU).
3. Итоговую таблицу и строки пропускной способности выше.

Этого достаточно, чтобы собрать разделы «Программное и аппаратное обеспечение», «Результаты» и «Выводы».